<table><tr>
<td style="background:#003057;color:#C29122;font-family:Georgia,serif;font-weight:700;
           font-size:15px;padding:8px 12px;border-radius:6px">GT</td>
<td style="padding-left:12px">
<b>NeuroAI: Models of the Brain and Mind</b> &nbsp;·&nbsp; PSYC 4690 / PSYC 6690<br>
<span style="color:#4a4a45">Comparing models and brains &nbsp;·&nbsp; N. Apurva Ratan Murty, PhD &nbsp;·&nbsp;
School of Psychological and Brain Sciences, Georgia Tech</span>
</td></tr></table>

# Tutorial 2 · Encoding Models

**Can we build each IT site out of AlexNet's units?**

RSA asked whether the two systems arrange images the same way. An encoding model asks something more direct:
take one IT site, and find a weighted sum of AlexNet units that reproduces its response to every image. Then
freeze those weights and test them on images that played no part in choosing them.

In this notebook you will:

1. fit a **ridge regression** from AlexNet features to all 449 IT sites at once,
2. score each site on **held-out images**,
3. watch what the **ridge penalty λ** actually does,
4. find which **layer** predicts IT best,
5. run the harder test: **hold out an entire category**.

Companion reading: the encoding-models tutorial page. Run the cells in order.

## 0 · Setup

Two things to do before anything else.

1. **Turn on a GPU.** `Runtime → Change runtime type → T4 GPU`. Everything here runs on CPU too, it is just slower.
2. **Put the data in your Google Drive.** Make a folder called `neuroai_tutorial` in *My Drive* and
   put `responses.npy` and `stimuli.zip` inside it. If you put it somewhere else, edit `DATA_DIR` below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/neuroai_tutorial'   # <-- edit if your folder is elsewhere

import os
assert os.path.exists(f'{DATA_DIR}/responses.npy'), f'responses.npy not found in {DATA_DIR}'
assert os.path.exists(f'{DATA_DIR}/stimuli.zip'),   f'stimuli.zip not found in {DATA_DIR}'
print('Found the data.')

In [ ]:
# Copy the images onto the Colab machine. Reading 1379 files straight from Drive is slow.
!unzip -q -o "$DATA_DIR/stimuli.zip" -d /content/ -x "__MACOSX/*"
STIM_DIR = '/content/stimuli'
print(len(os.listdir(STIM_DIR)), 'images unzipped to', STIM_DIR)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image

# ---- course plotting style -------------------------------------------------
NAVY, GOLD, RED, TEAL, GREY = '#003057', '#C29122', '#be3a2a', '#0c7a5e', '#4a4a45'
plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.edgecolor':'#c8c8c4', 'axes.linewidth':0.9,
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.titlesize':12, 'axes.titleweight':'semibold', 'axes.titlepad':10,
    'axes.labelsize':10.5, 'axes.labelcolor':'#242420',
    'xtick.color':GREY, 'ytick.color':GREY, 'xtick.labelsize':9.5, 'ytick.labelsize':9.5,
    'xtick.major.size':3.5, 'ytick.major.size':3.5,
    'legend.frameon':False, 'legend.fontsize':9.5,
    'grid.color':'#ececea', 'grid.linewidth':0.8,
    'font.size':10.5, 'figure.dpi':110, 'savefig.bbox':'tight',
})
RDM_CMAP = LinearSegmentedColormap.from_list('rdm', ['#f7f7f5', '#cfd8e0', NAVY])
print('numpy', np.__version__)

In [ ]:
def zscore(A, axis=0):
    A = np.asarray(A, float)
    mu, sd = A.mean(axis, keepdims=True), A.std(axis, keepdims=True)
    return (A - mu) / np.where(sd < 1e-12, 1.0, sd)

def pearson_cols(A, B):
    """Column-wise Pearson r between two (n, k) arrays -> (k,)."""
    A, B = A - A.mean(0), B - B.mean(0)
    den = np.sqrt((A*A).sum(0) * (B*B).sum(0))
    return np.where(den < 1e-12, 0.0, (A*B).sum(0) / den)

## 1 · The data

`responses.npy` holds recordings from **449 sites in macaque inferotemporal (IT) cortex**, measured while the
animal viewed **1379 images**. Each site has already been z-scored across images, so a value is
"how far above or below this site's average response was this image", in units of its own standard deviation.

The images are `im0001.png` … `im1379.png`, and **column *j* of `responses.npy` is image `im{j+1:04d}.png`**.
That correspondence is the whole basis of everything below, so we check it explicitly.

The stimulus set has two parts: the first 447 images are **faces** (human and monkey) and the remaining 932 are
**objects**. We will use that split throughout, both to read the plots and to build a harder test at the end.

In [ ]:
RESP = np.load(f'{DATA_DIR}/responses.npy')          # (449 sites, 1379 images)
IT   = RESP.T.astype(np.float64)                     # (1379 images, 449 sites)  <- we work image-major
N_IMG, N_SITE = IT.shape

IMG_PATHS = [f'{STIM_DIR}/im{i:04d}.png' for i in range(1, N_IMG + 1)]
assert all(os.path.exists(p) for p in IMG_PATHS), 'image / response count mismatch'

N_FACES   = 447
is_face   = np.arange(N_IMG) < N_FACES
labels    = (~is_face).astype(int)                   # 0 = face, 1 = object
CAT_NAMES = ['faces', 'objects']
CAT_COLS  = [RED, NAVY]
BOUNDS    = [0, N_FACES, N_IMG]

print(f'{N_SITE} IT sites  x  {N_IMG} images')
print(f'{is_face.sum()} faces, {(~is_face).sum()} objects')
print('each site is z-scored across images:  mean %.1e,  sd %.3f' % (IT[:,0].mean(), IT[:,0].std()))

In [ ]:
def montage(indices, ncol=8, size=92, title=''):
    rows = int(np.ceil(len(indices) / ncol))
    sheet = Image.new('RGB', (ncol*size, rows*size), 'white')
    for k, i in enumerate(indices):
        sheet.paste(Image.open(IMG_PATHS[i]).resize((size, size)), ((k % ncol)*size, (k // ncol)*size))
    fig, ax = plt.subplots(figsize=(ncol*0.85, rows*0.85 + 0.4))
    ax.imshow(sheet); ax.axis('off'); ax.set_title(title)
    return fig

rng = np.random.default_rng(0)
montage(np.r_[rng.choice(np.flatnonzero(is_face), 8, replace=False),
              rng.choice(np.flatnonzero(~is_face), 8, replace=False)],
        title='Top row: faces (images 1–447).   Bottom row: objects (images 448–1379).')
plt.show()

## 2 · AlexNet

We load AlexNet with its ImageNet-trained weights and tap **eight layers**: the five convolutional stages
(after their ReLUs) and the three fully-connected ones. This is the model side of the comparison.

Two practical points:

* The raw activations are enormous — `conv1` alone gives 64 × 55 × 55 = 193,600 numbers per image. We keep a
  **fixed random sample of 2048 units per layer**. Random subsampling does not privilege any particular units,
  and both RSA and ridge regression are stable to it. Raise `MAX_UNITS` if you want to check that for yourself.
* Nothing about AlexNet is adjusted using the neural data. The network is frozen. This matters:
  it is the same logic as Yamins et al. (2014), where the model was chosen for task performance and only
  *then* compared to the brain.

In [ ]:
import torch, torchvision.transforms as T
from torchvision.models import alexnet, AlexNet_Weights

device = 'cuda' if torch.cuda.is_available() else 'cpu'
net = alexnet(weights=AlexNet_Weights.IMAGENET1K_V1).to(device).eval()
print('AlexNet loaded on', device)

# the eight layers we read out (ReLU outputs for conv1-5 and fc6-7, logits for fc8)
TAPS = {'conv1': net.features[1],  'conv2': net.features[4],  'conv3': net.features[7],
        'conv4': net.features[9],  'conv5': net.features[11],
        'fc6'  : net.classifier[2],'fc7'  : net.classifier[5],'fc8'  : net.classifier[6]}
LAYERS = list(TAPS)

preprocess = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),   # ImageNet statistics
])

In [ ]:
MAX_UNITS = 2048        # units kept per layer
BATCH     = 64

def extract_features(paths, max_units=MAX_UNITS, batch=BATCH, seed=0):
    buf, out, keep = {}, {k: [] for k in TAPS}, {}
    hooks = [m.register_forward_hook(
                lambda mod, inp, o, k=k: buf.__setitem__(k, o.detach().flatten(1).cpu()))
             for k, m in TAPS.items()]
    rng = np.random.default_rng(seed)
    with torch.no_grad():
        for s in range(0, len(paths), batch):
            x = torch.stack([preprocess(Image.open(p).convert('RGB'))
                             for p in paths[s:s+batch]]).to(device)
            net(x)
            for k, v in buf.items():
                v = v.numpy()
                if k not in keep:
                    n = v.shape[1]
                    keep[k] = np.sort(rng.choice(n, min(max_units, n), replace=False))
                out[k].append(v[:, keep[k]])
            if (s // batch) % 5 == 0:
                print(f'  {min(s+batch, len(paths))}/{len(paths)} images', end='\r')
    for h in hooks: h.remove()
    return {k: np.concatenate(v).astype(np.float32) for k, v in out.items()}

CACHE = f'{DATA_DIR}/alexnet_features_{MAX_UNITS}.npz'
if os.path.exists(CACHE):
    F = dict(np.load(CACHE));  print('loaded cached features from Drive')
else:
    F = extract_features(IMG_PATHS)
    np.savez_compressed(CACHE, **F);  print('\nextracted and cached to Drive')

for k in LAYERS:
    print(f'  {k:>6}: {F[k].shape}')

## 3 · Splitting the images

Everything rests on this cell. The model will be fitted on the training images and **never** scored on them.
We hold out 20% of the images at random, keeping the face/object proportions the same on both sides so the
held-out set is not accidentally all objects.

In [ ]:
rng = np.random.default_rng(2026)
train_idx, test_idx = [], []
for c in (0, 1):                                   # stratify by category
    ids = np.flatnonzero(labels == c); rng.shuffle(ids)
    cut = int(0.8 * len(ids))
    train_idx += list(ids[:cut]); test_idx += list(ids[cut:])
train_idx, test_idx = np.sort(train_idx), np.sort(test_idx)

print(f'train: {len(train_idx)} images   ({(labels[train_idx]==0).sum()} faces)')
print(f'test : {len(test_idx)} images   ({(labels[test_idx]==0).sum()} faces)')

## 4 · Fitting the encoding model

`X` holds AlexNet features (one row per image, one column per unit) and `Y` holds the measured IT responses
(one row per image, one column per site). We want the weights `W` with `Ŷ = XW`.

Two details from the tutorial matter here:

* **Regularization.** We have 2048 features and only ~1100 training images, so an unpenalized fit has far too
  much freedom. Ridge adds a cost for large weights. `RidgeCV` picks λ by cross-validation *inside the training
  set*, and `alpha_per_target=True` lets every IT site get its own λ.
* **Standardize on training data only.** The scaler is part of the pipeline, so its mean and standard deviation
  are computed from training images alone. Computing them over all images would leak information about the
  held-out set.

In [ ]:
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

LAMBDAS = np.logspace(1, 7, 13)

def fit_encoding(X, Y, train, test, lambdas=LAMBDAS):
    """Fit all sites at once; return held-out predictions, measured values and per-site r."""
    model = make_pipeline(StandardScaler(),
                          RidgeCV(alphas=lambdas, alpha_per_target=True))
    model.fit(X[train], Y[train])
    Y_hat = model.predict(X[test])
    return Y_hat, Y[test], pearson_cols(Y_hat, Y[test]), model

LAYER = 'fc6'                                     # start with one layer
Y_hat, Y_true, r_sites, model = fit_encoding(F[LAYER].astype(np.float64), IT, train_idx, test_idx)

print(f'layer {LAYER}:  median held-out r = {np.median(r_sites):.3f}')
print(f'            best site r = {r_sites.max():.3f},  worst = {r_sites.min():.3f}')
print(f'            sites with r > 0.5: {(r_sites > 0.5).sum()} / {N_SITE}')

### One score per site

There is no single number for "how well AlexNet predicts IT" — there are 449 of them, one per site. The
distribution is the result; the median is a summary of it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.0))

axes[0].hist(r_sites, bins=34, color=NAVY, alpha=.88)
axes[0].axvline(np.median(r_sites), color=GOLD, lw=2.2,
                label=f'median = {np.median(r_sites):.3f}')
axes[0].set_xlabel('held-out correlation  r'); axes[0].set_ylabel('number of IT sites')
axes[0].set_title(f'How well is each site predicted?   ({LAYER})'); axes[0].legend()
axes[0].yaxis.grid(True); axes[0].set_axisbelow(True)

b = int(np.argmax(r_sites))
axes[1].scatter(Y_hat[:, b], Y_true[:, b], s=15, alpha=.5, linewidths=0,
                c=[CAT_COLS[l] for l in labels[test_idx]])
lo = min(Y_hat[:, b].min(), Y_true[:, b].min()); hi = max(Y_hat[:, b].max(), Y_true[:, b].max())
axes[1].plot([lo, hi], [lo, hi], color=GREY, lw=1.2, ls='--')
axes[1].set_xlabel('predicted response'); axes[1].set_ylabel('measured response')
axes[1].set_title(f'Best-predicted site (#{b}),  r = {r_sites[b]:.3f}')
axes[1].yaxis.grid(True); axes[1].set_axisbelow(True)
fig.tight_layout(); plt.show()

The scatter is one synthetic neuron against one real one, on images the weights never saw. Points are coloured
by category — red for faces, navy for objects — which usually makes it obvious what the best-predicted sites
are doing.

## 5 · What λ actually does

This is the figure worth staring at. We sweep the ridge penalty from almost nothing to very large, and score
the *same* model on the training images and on the held-out images.

Small λ: near-perfect on training images, poor on held-out ones. The model has enough freedom to fit the noise.
Large λ: the weights are crushed toward zero and the model underfits both. Somewhere in between is the λ that
`RidgeCV` picks for you.

In [ ]:
sweep = np.logspace(0, 8, 17)
tr_med, te_med = [], []
Xl = F[LAYER].astype(np.float64)
for lam in sweep:
    m = make_pipeline(StandardScaler(), Ridge(alpha=lam)).fit(Xl[train_idx], IT[train_idx])
    te_med.append(np.median(pearson_cols(m.predict(Xl[test_idx]),  IT[test_idx])))
    tr_med.append(np.median(pearson_cols(m.predict(Xl[train_idx]), IT[train_idx])))

fig, ax = plt.subplots(figsize=(7.0, 4.0))
ax.semilogx(sweep, tr_med, 'o-', color=GOLD, lw=2, ms=5, label='training images')
ax.semilogx(sweep, te_med, 'o-', color=NAVY, lw=2, ms=5, label='held-out images')
ax.axvline(sweep[int(np.argmax(te_med))], color=RED, lw=1.2, ls='--',
           label=f'best λ ≈ {sweep[int(np.argmax(te_med))]:.0e}')
ax.set_xlabel('ridge penalty  λ'); ax.set_ylabel('median r across the 449 sites')
ax.set_title('Why only the held-out score counts')
ax.yaxis.grid(True); ax.set_axisbelow(True); ax.legend(loc='lower left')
plt.show()

## 6 · Every layer

Now the comparison that mirrors the classic result: fit a separate encoding model from each AlexNet layer and
ask which one gives the best basis for IT. This cell fits 8 × 449 encoding models, so give it a couple of minutes.

In [ ]:
enc_r = {}
for L in LAYERS:
    _, _, r, _ = fit_encoding(F[L].astype(np.float64), IT, train_idx, test_idx)
    enc_r[L] = r
    print(f'  {L:>6}   median r = {np.median(r):.3f}')

med = np.array([np.median(enc_r[L]) for L in LAYERS])

def layer_bars(names, vals, title, ylab, ax=None):
    solo = ax is None
    if solo: fig, ax = plt.subplots(figsize=(7.6, 3.8))
    best = int(np.argmax(vals))
    ax.bar(np.arange(len(vals)), vals, width=.66,
           color=[GOLD if i == best else NAVY for i in range(len(vals))])
    for i, v in enumerate(vals):
        ax.text(i, v + max(vals)*.025, f'{v:.3f}', ha='center', fontsize=9,
                fontweight='600', color=GOLD if i == best else GREY)
    ax.set_xticks(np.arange(len(names))); ax.set_xticklabels(names)
    ax.set_ylabel(ylab); ax.set_title(title)
    ax.set_ylim(0, max(vals)*1.18); ax.yaxis.grid(True); ax.set_axisbelow(True)
    return fig if solo else ax

layer_bars(LAYERS, med, 'Which AlexNet layer best predicts IT?',
           'median held-out r  (449 sites)')
plt.show()

In [ ]:
# the full distributions, not just the medians
fig, ax = plt.subplots(figsize=(8.2, 4.2))
parts = ax.violinplot([enc_r[L] for L in LAYERS], showmedians=True, widths=.82)
for pc in parts['bodies']:
    pc.set_facecolor(NAVY); pc.set_alpha(.35); pc.set_edgecolor('none')
for key in ('cmedians', 'cbars', 'cmins', 'cmaxes'):
    parts[key].set_color(NAVY); parts[key].set_linewidth(1.2)
parts['cmedians'].set_color(GOLD); parts['cmedians'].set_linewidth(2.4)
ax.set_xticks(np.arange(1, len(LAYERS)+1)); ax.set_xticklabels(LAYERS)
ax.axhline(0, color=GREY, lw=.8, ls=':')
ax.set_ylabel('held-out r'); ax.set_title('All 449 sites, every layer')
ax.yaxis.grid(True); ax.set_axisbelow(True)
plt.show()

## 7 · The harder test: hold out a whole category

The random split above still shows the model plenty of faces during fitting. A stronger test is to fit the
weights on **objects only** and then ask them to predict responses to **faces**, a kind of image the encoding
model has never been fitted on.

If the score survives, the weights describe something general about how this site reads out AlexNet's features.
If it collapses, the earlier score was partly a matter of having seen similar images during fitting.

In [ ]:
obj_train, face_test = np.flatnonzero(~is_face), np.flatnonzero(is_face)

# reuses enc_r from the previous section, so only the category split is fitted here
rows = []
for L in LAYERS:
    _, _, r_cat, _ = fit_encoding(F[L].astype(np.float64), IT, obj_train, face_test)
    rows.append((np.median(enc_r[L]), np.median(r_cat)))
    print(f'  {L:>6} done', end='\r')
rows = np.array(rows)

x = np.arange(len(LAYERS)); w = 0.38
fig, ax = plt.subplots(figsize=(8.6, 4.0))
ax.bar(x - w/2, rows[:, 0], w, color=NAVY, label='random split (held-out images)')
ax.bar(x + w/2, rows[:, 1], w, color=GOLD, label='category split (fit objects → predict faces)')
ax.set_xticks(x); ax.set_xticklabels(LAYERS)
ax.set_ylabel('median held-out r'); ax.set_title('How far do the weights generalize?')
ax.axhline(0, color=GREY, lw=.8); ax.yaxis.grid(True); ax.set_axisbelow(True); ax.legend()
plt.show()

for L, (a, b) in zip(LAYERS, rows):
    pct = f'{100*b/a:.0f}% retained' if a > 0.05 else '(baseline too low to compare)'
    print(f'  {L:>6}   random {a:.3f}   category {b:.3f}   {pct}')

## · A word on the noise ceiling

Every score in this notebook is a raw correlation, and raw correlations are hard to read on their own.
Measure the same IT site twice and the two measurements will not agree perfectly. No model can be expected to
predict the part of a response that does not even replicate, so the honest question is never "how close to 1.0
is this score?" but "how close to the site's own reliability is it?"

That reliability is the **noise ceiling**, and computing it needs *repeated* measurements of the same images —
which this dataset does not include. So we report raw numbers here, and you should read them as
*relative* comparisons between layers rather than as absolute statements about how good AlexNet is.

With repeat data in hand, the ceiling is short to compute:

```python
# run1, run2: (n_images, n_sites) from two independent repetitions
ceiling    = pearson_cols(run1[test], run2[test])          # per-site reliability
normalized = r / np.sqrt(np.maximum(ceiling, 1e-6))        # score read against the ceiling
```

## · Exercises

1. **Read the weights.** For the best-predicted site, pull `model[-1].coef_[b]` and look at the distribution of
   weights. Are a few units doing the work, or many? Refit with a much larger λ and look again.

2. **How many images do you need?** Refit using 10%, 25%, 50% and 100% of the training images and plot median
   held-out r against training-set size. Where does the curve flatten?

3. **How many units do you need?** Re-extract with `max_units` of 128, 512 and 2048 and compare. Does the layer
   ordering change, or just the overall level?

4. **An untrained control.** Repeat the layer sweep with `alexnet(weights=None)`. A random network is still a
   rich set of nonlinear image features, so it will not score zero — which is exactly why the comparison to a
   *trained* network is the informative one.

5. **Connect the two notebooks.** Take the predicted responses `Y_hat` for the best layer, build an RDM from
   them, and correlate it with the RDM of the measured responses on the same held-out images. This is the
   analysis in Figure 5 of the encoding-models page.